# 10 — Methods Overview

This notebook describes the three methods used to assign animals as
BE or SC and to estimate their parameters. No data is loaded here —
see NB 11–13 for validation and NB 20–22 for real results.

**Methods:**
1. Grid-search cross-validation (GS-CV) — established benchmark
2. Static SBI (amortised SNPE) — proposed complement
3. Dynamic SBI — per-animal trajectory estimation

**Model comparison:** Wilcoxon signed-rank test on paired CV errors,
applied identically regardless of which method produced them.

In [1]:
%matplotlib inline
from shared_setup import *
apply_style()

## 1. Fitting targets

Both GS and SBI evaluate model fit using the same two targets,
computed from pooled expert sessions.

### Update matrix (UM)

The UM captures how the previous trial's stimulus and outcome shift
choice probability on the current trial. For an 8 × 8 grid of
(previous stimulus bin) × (current stimulus bin), each cell is the
difference in P(choose B) between post-correct and post-error trials:

$$
\text{UM}[i,j] = P(B \mid s_t \in \text{bin}_j,\, s_{t-1} \in \text{bin}_i,\,\text{correct}_{t-1})
\;-\; P(B \mid s_t \in \text{bin}_j,\, s_{t-1} \in \text{bin}_i,\,\text{error}_{t-1})
$$

The UM is the primary fitting target because BE and SC produce
qualitatively different UM patterns (see NB 02):
- **BE:** strongest serial dependence near the boundary
- **SC:** strongest serial dependence at the stimulus extremes

### Conditional psychometric curve (CP)

The CP conditions the psychometric function on the previous stimulus
bin. Each bin yields a separate psychometric curve; the shift in
midpoint (PSE) across bins is the serial dependence signature.
Used as a robustness check alongside the UM.

### Error metric

For both targets, model fit is measured as the mean squared error
between empirical and model-predicted matrices:

$$
\text{MSE} = \frac{1}{N_{\text{valid}}} \sum_{i,j \in \text{valid}} \left( M^{\text{emp}}_{i,j} - M^{\text{sim}}_{i,j} \right)^2
$$

where `valid` cells have ≥ 1 trial in both the empirical and
simulated matrices.

## 2. Grid-search cross-validation (GS-CV)

The established benchmark from Menichini et al. 2025.

### Procedure

1. **Pool** all expert sessions for one animal into a single
   stimulus/choice/category array.

2. **Split** into 2 folds by session blocks (first half / second half),
   preserving temporal order.

3. **Grid search** over the model's parameter space. For each grid
   point θ:
   - Simulate the model on the training fold's stimuli
   - Compute the simulated UM (or CP)
   - MSE against the training fold's empirical UM

4. **Best parameters** = grid point with lowest training MSE.

5. **Test error** = simulate with best parameters on the held-out fold's
   stimuli, compute MSE against the held-out fold's empirical UM.

6. **Repeat** across 32–64 random seeds (randomising the simulation
   stochasticity). Each seed yields one test error.

7. **Collect** the per-seed test errors as `cv_errors`.

### Parameters

| Setting | Value | Rationale |
|---------|-------|----------|
| Folds | 2 | Session-block split (first/second half) |
| Seeds | 32–64 | Stabilise stochastic simulation |
| Burn-in | 1000 trials | Let beliefs converge before analysis |
| Grid | 8 × 8 | Per model parameter |
| `fit_target` | `update_matrix` (primary), `conditional_psych` (robustness) |

### Strengths and limitations

**Strengths:** Conceptually simple, true held-out evaluation,
reproducible, fast (~1–2 min per animal).

**Limitations:** Grid discretisation means the best-fit parameters
are approximate. Poor parameter recovery for continuous parameters
(especially `eta_learning`). Cannot extend to multi-session or
temporal estimation.

## 3. Static SBI (amortised SNPE)

### Training (once per model type)

1. **Define prior** $p(\theta)$: uniform box over each model parameter.

2. **Simulate** $N = 50{,}000$ synthetic datasets:
   - Sample $\theta \sim p(\theta)$
   - Simulate the model on a curriculum-matched stimulus sequence
     (e.g. 15 uniform sessions × 350 trials)
   - Chain the model state across sessions (the state from session
     $k$ becomes the initial state for session $k+1$)
   - Compute summary statistics per session, concatenate

3. **Train** a neural density estimator (masked autoregressive flow)
   via SNPE (Papamakarios et al. 2019) to approximate
   $q_\phi(\theta \mid \mathbf{x})$ where $\mathbf{x}$ is the
   concatenated summary statistics.

The trained network is **amortised**: it can produce a posterior for
any animal in ~1 second, without retraining.

### Summary statistics

Per session, we compute ~20 heuristic statistics: accuracy,
psychometric parameters (PSE, slope, lapse rates), history effects
(recency, stimulus recency), choice patterns (win-stay, lose-shift,
perseveration, choice entropy), side bias, and stimulus sensitivity.

These are deliberately **not** the UM or CP — those are reserved as
independent evaluation targets. The summary statistics inform the
posterior; the UM/CP test whether the posterior's predictions
generalise.

### Conditioning (per animal)

1. Compute the animal's observed summary statistics
   $\mathbf{x}_{\text{obs}}$ from its expert sessions.

2. Sample from the amortised posterior:
   $\theta_1, \ldots, \theta_K \sim q_\phi(\theta \mid \mathbf{x}_{\text{obs}})$

3. Point estimate = posterior median.

### Held-out CV (matching GS protocol)

To produce CV errors directly comparable to GS:

1. **Split** sessions into first half / second half (same as GS).

2. For each fold pair (train, test):
   - Compute summary stats on **train** sessions
   - Condition posterior on train stats → samples
   - For each posterior sample:
     - Simulate model on **test** fold stimuli
     - Compute UM/CP on simulated choices
     - MSE against test fold empirical UM/CP
   - Mean across samples = one fold error

3. Collect fold errors as `cv_errors`.

### Strengths and limitations

**Strengths:** Continuous parameter estimation (not grid-discretised),
full posterior (uncertainty quantification), fast conditioning (~1s),
extends naturally to multi-session and dynamic estimation.

**Limitations:** Requires upfront training (~30 min), posterior quality
depends on summary statistic choice, potential amortisation gap if
real data falls outside the simulated training distribution.

## 4. Dynamic SBI

For estimating how parameters evolve across sessions (learning
trajectories, post-shift adaptation). This is a per-animal method —
a new network is trained for each animal.

### Temporal link functions

Each parameter $\theta_k$ at session $k$ is linked to session $k-1$
via a **link specification**:

| Link | Model | Use case |
|------|-------|----------|
| **Constant** | $\theta_k = \theta_0 \;\forall k$ | Perceptual params (stable) |
| **RandomWalk** | $\theta_k = \theta_{k-1} + \epsilon$, $\epsilon \sim \mathcal{N}(0, \sigma^2_{\text{drift}})$ | Smooth learning |
| **GP** | $\theta_{1:K} \sim \mathcal{GP}(\mu, k)$ | Non-parametric dynamics |

Typically: perceptual parameters ($\sigma_{\text{percep}}$,
$A_{\text{repulsion}}$) are held constant, while learning parameters
($\eta_{\text{learning}}$ for BE, $\gamma$ for SC) vary.

### Procedure

1. Build a **multi-session prior** combining constant and varying
   parameter priors.

2. Build a **multi-session simulator** that chains model state across
   sessions, using the animal's actual stimulus sequences.

3. Train SNPE ($N \approx 30{,}000$ sims, ~20 min per animal).

4. Extract **trajectories**: posterior median and credible intervals
   for each parameter at each session.

### Evaluation

Dynamic SBI uses **posterior predictive check** (PPC) rather than
true cross-validation (retraining per fold is too expensive):

For each session $k$:
- Take the trajectory's median parameters at session $k$
- Simulate on session $k$'s stimuli
- Compute UM, MSE against empirical

These PPC errors (`cv_type='ppc'`) are **not** directly comparable
to the held-out errors from GS and static SBI (`cv_type='held_out'`).
They measure goodness-of-fit, not generalisation.

### Strengths and limitations

**Strengths:** Captures parameter dynamics, per-session resolution,
uncertainty quantification on trajectories, flexible link functions.

**Limitations:** Expensive (per-animal training), RandomWalk cannot
represent discontinuous strategy switches (motivating the SLDS +
per-state static selection pipeline), PPC rather than true CV.

## 5. Model comparison protocol

The same comparison protocol applies regardless of whether errors
came from GS or SBI.

### Per-method assignment

For one method × one fit target:

1. Run the method for both BE and SC on the same animal.
2. Collect `cv_errors` from each.
3. **Wilcoxon signed-rank test** on paired errors
   (BE errors vs SC errors, same seeds/folds).
4. Winner = model with lower mean error.
5. Significant if $p < 0.05$.

### Four-method consensus

We run four method × target combinations:

| | Update matrix | Conditional psych |
|---|---|---|
| **Grid search** | GS-UM | GS-CP |
| **Static SBI** | SBI-UM | SBI-CP |

**Consensus rule:** Majority vote across the four methods.
Animals where fewer than 3 methods agree are classified as
**unclear**.

### Current cohort

From 23 animals: **11 BE**, **2 SC**, **9 unclear** (still training).

## 6. Pipeline summary

The full analysis pipeline for one animal:

```
Raw sessions
  │
  ├─ select_sessions(preset='expert_uniform')
  ├─ filter_trials()
  │
  ├─── Grid Search ──────────────── cv_errors (held_out) ─┐
  │    grid_search_cv(sessions)                            │
  │    × BE and SC                                         ├── compare_results()
  │                                                        │   → winner, p_value
  ├─── Static SBI ───────────────── cv_errors (held_out) ─┘
  │    AmortisedSBI.fit(sessions)                          
  │    × BE and SC                                         
  │                                                        
  ├─── Consensus ─── majority_vote([gs_um, gs_cp,
  │                                  sbi_um, sbi_cp])
  │                   → 'BE' | 'SC' | 'unclear'
  │
  └─── Dynamic SBI ──────────────── trajectories + ppc_errors
       SBIFitter(fitting_data, param_links)
       (applied to winning model only)
```

### Scripts

| Script | What it does |
|--------|-------------|
| `scripts/grid_search.py` | GS-CV for one animal × model |
| `scripts/sbi.py train` | Train amortised SNPE |
| `scripts/sbi.py static` | Condition + held-out CV |
| `scripts/sbi.py dynamic` | Per-animal trajectory estimation |
| `scripts/compare_models.py` | Wilcoxon on BE vs SC errors |